# AI API Aggregator — Colab Demo

OpenAI ve Google Gemini API'lerini tek bir FastAPI endpoint'i arkasında birleştiren projenin Colab sürümü.
Hücreler `app/` klasöründeki dosyaları `%%writefile` ile oluşturur, sonra sunucuyu ngrok üzerinden yayına açar.

**Çalıştırmadan önce:** sol menüdeki 🔑 Secrets bölümüne `INTERNAL_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` ve `NGROK_AUTH_TOKEN` ekleyin.

In [ ]:
# @title Hücre 1: Gerekli Kütüphaneler ve Ortam Kurulumu
# API sunucusu, tünelleme ve veri doğrulama için gerekli paketler.
!pip install fastapi uvicorn "httpx[http2]" pyngrok pydantic-settings

In [ ]:
# @title Hücre 2: Proje Dizin Yapısını Oluşturma
# Temiz bir mimari için klasörleri ve paket dosyalarını oluşturuyoruz.
import os

folders = [
    "app/core",
    "app/models",
    "app/routers",
    "app/services",
    "app/providers"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    with open(f"{folder}/__init__.py", "w") as f:
        pass # Boş init dosyası

with open("app/__init__.py", "w") as f:
    pass

print("✅ Proje dizin yapısı oluşturuldu.")

In [ ]:
# @title Hücre 3: Gizli Anahtarlar (.env)
# [!!! ÖNEMLİ !!!] Anahtarları buraya YAZMAYIN.
# Colab'ın sol menüsündeki 🔑 "Secrets" bölümüne şu isimlerle ekleyin:
#   INTERNAL_API_KEY, GEMINI_API_KEY, OPENAI_API_KEY, NGROK_AUTH_TOKEN
# Bu hücre .env dosyasını o değerlerden oluşturur.
from google.colab import userdata

with open(".env", "w") as f:
    for name in ["INTERNAL_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY"]:
        try:
            f.write(f'{name}="{userdata.get(name)}"\n')
        except Exception:
            print(f"⚠️ {name} Secrets içinde bulunamadı, atlanıyor.")

print("✅ .env oluşturuldu.")

In [ ]:
# @title Hücre 4: Ayar Yöneticisi (Config)
# .env dosyasındaki verileri okur ve uygulama genelinde kullanılabilir hale getirir.
%%writefile app/core/config.py
from pydantic_settings import BaseSettings
from typing import Optional

class Settings(BaseSettings):
    INTERNAL_API_KEY: str
    GEMINI_API_KEY: Optional[str] = None
    OPENAI_API_KEY: Optional[str] = None

    # Gelecekte eklenecek özellikler için yer tutucular
    # TTS_ENABLED: bool = False
    # IMAGE_GEN_ENABLED: bool = False

    class Config:
        env_file = ".env"
        extra = 'ignore' # Bilinmeyen değişkenleri görmezden gel

settings = Settings()

In [ ]:
# @title Hücre 5: Güvenlik Katmanı (Security)
# Gelen isteklerin yetkili olup olmadığını kontrol eder.
%%writefile app/core/security.py
from fastapi import Security, HTTPException, status
from fastapi.security import APIKeyHeader
from app.core.config import settings

# Header'da 'Authorization: Bearer <key>' beklenir
api_key_header = APIKeyHeader(name="Authorization", auto_error=False)

async def get_api_key(key: str = Security(api_key_header)):
    """
    Kullanıcının gönderdiği API anahtarını doğrular.
    """
    if not key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Kimlik doğrulama hatası: 'Authorization' başlığı eksik."
        )

    # 'Bearer ' öneki kontrolü
    parts = key.split()
    if len(parts) != 2 or parts[0].lower() != "bearer":
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Format hatası: Anahtar 'Bearer <API_KEY>' formatında olmalıdır."
        )

    # Anahtar kontrolü (Gerçek senaryoda veritabanından bakılır)
    if parts[1] == settings.INTERNAL_API_KEY:
        return True
    else:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Yetkisiz erişim: Geçersiz API Anahtarı."
        )

In [ ]:
# @title Hücre 6: Sohbet Veri Modelleri (Text Generation)
# Kullanıcıdan gelen ve kullanıcıya giden verinin şeklini belirler.
# "Kolay Kullanım" için birçok alan opsiyonel (Optional) yapılmıştır.
%%writefile app/models/chat.py
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any, Union

# Standart bir mesaj yapısı
class Message(BaseModel):
    role: str = Field(..., description="Mesajı gönderen: 'user', 'system' veya 'assistant'")
    content: str = Field(..., description="Mesajın metin içeriği")

# Gelen İstek Modeli (Esnek Yapı)
class ChatRequest(BaseModel):
    # Basit mod için tekil prompt (Opsiyonel)
    prompt: Optional[str] = Field(None, description="Hızlı kullanım için tekil metin girişi.")

    # Gelişmiş mod için mesaj geçmişi (Opsiyonel)
    messages: Optional[List[Message]] = Field(None, description="Sohbet geçmişi listesi.")

    # Model seçimi (Opsiyonel - Default: 'oto')
    model: Optional[str] = Field("oto", description="Kullanılacak AI modeli.")

    # Ayarlar (Opsiyonel - Default değerler serviste atanır)
    stream: Optional[bool] = Field(False, description="Yanıtın akış (stream) olarak gelip gelmeyeceği.")
    temperature: Optional[float] = Field(1.0, description="Yaratıcılık seviyesi (0.0 - 2.0).")
    max_tokens: Optional[int] = Field(None, description="Maksimum üretim uzunluğu.")

    # İleri seviye ayarlar (JSON Mode, TopP vb.)
    json_mode: Optional[bool] = Field(False, description="Çıktının JSON formatında olmasını zorlar.")

# Dönüş Modeli (Standartlaştırılmış)
class ChatResponse(BaseModel):
    model_used: str
    content: str
    role: str = "assistant"
    finish_reason: Optional[str] = None
    # usage: Optional[Dict[str, int]] = None # Token kullanımı (İleride eklenecek)

In [ ]:
# @title Hücre 7: Temel Sağlayıcı Şablonu (Base Provider)
# Tüm AI modellerinin uyması gereken ortak kurallar.
%%writefile app/providers/base.py
from abc import ABC, abstractmethod
from app.models.chat import ChatRequest, ChatResponse
import httpx
from typing import Dict, Any, AsyncGenerator

class BaseProvider(ABC):
    def __init__(self, client: httpx.AsyncClient):
        self.client = client

    @abstractmethod
    def get_url(self, request: ChatRequest) -> str:
        """API uç noktası URL'sini döndürür."""
        pass

    @abstractmethod
    def get_headers(self) -> Dict[str, str]:
        """Yetkilendirme başlıklarını döndürür."""
        pass

    @abstractmethod
    def convert_request(self, request: ChatRequest) -> Dict[str, Any]:
        """Bizim standart isteğimizi, sağlayıcının formatına çevirir."""
        pass

    @abstractmethod
    def convert_response(self, provider_response: Dict[str, Any], model_name: str) -> ChatResponse:
        """Sağlayıcının cevabını, bizim standart formatımıza çevirir."""
        pass

    @abstractmethod
    async def stream_chat(self, request: ChatRequest) -> AsyncGenerator[str, None]:
        """Akışlı (Streaming) yanıt üretir."""
        yield "" # Abstract method placeholder

In [ ]:
# @title Hücre 8: Google Gemini Sağlayıcısı (Gelişmiş Hata Yönetimi)
%%writefile app/providers/gemini.py
from app.providers.base import BaseProvider
from app.models.chat import ChatRequest, ChatResponse
from app.core.config import settings
from typing import Dict, Any, AsyncGenerator
import json

class GeminiProvider(BaseProvider):
    def get_headers(self) -> Dict[str, str]:
        return {"Content-Type": "application/json"}

    def get_url(self, request: ChatRequest) -> str:
        if not settings.GEMINI_API_KEY:
            raise ValueError("HATA: .env dosyasında GEMINI_API_KEY eksik!")

        base_url = f"https://generativelanguage.googleapis.com/v1beta/models/{request.model}:"
        if request.stream:
            return f"{base_url}streamGenerateContent?key={settings.GEMINI_API_KEY}&alt=sse"
        return f"{base_url}generateContent?key={settings.GEMINI_API_KEY}"

    def convert_request(self, request: ChatRequest) -> Dict[str, Any]:
        contents = []
        system_instruction = None

        source_messages = request.messages if request.messages else []
        for msg in source_messages:
            if msg.role == "system":
                system_instruction = {"parts": [{"text": msg.content}]}
            elif msg.role == "user":
                contents.append({"role": "user", "parts": [{"text": msg.content}]})
            elif msg.role == "assistant":
                contents.append({"role": "model", "parts": [{"text": msg.content}]})

        # İçerik boşsa varsayılan ekle
        if not contents:
             contents.append({"role": "user", "parts": [{"text": "Hello"}]})

        # Güvenlik Ayarları (Her şeyi serbest bırak)
        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
        ]

        generation_config = {
            "temperature": request.temperature,
            "maxOutputTokens": request.max_tokens,
        }

        if request.json_mode:
            generation_config["responseMimeType"] = "application/json"

        payload = {
            "contents": contents,
            "generationConfig": generation_config,
            "safetySettings": safety_settings
        }

        if system_instruction:
            payload["systemInstruction"] = system_instruction

        return payload

    def convert_response(self, provider_response: Dict[str, Any], model_name: str) -> ChatResponse:
        # --- GELİŞMİŞ HATA YÖNETİMİ ---
        # 1. 'candidates' kontrolü
        if not provider_response.get('candidates'):
            # Güvenlik filtresi (promptFeedback) kontrolü
            feedback = provider_response.get('promptFeedback')
            if feedback:
                block_reason = feedback.get('blockReason', 'UNKNOWN')
                # Hata fırlatmak yerine, güvenli bir "Ret" mesajı dönüyoruz
                return ChatResponse(
                    model_used=model_name,
                    content=f"⚠️ Model yanıt vermeyi reddetti (Güvenlik). Sebep: {block_reason}",
                    finish_reason="safety"
                )

            error = provider_response.get('error')
            if error:
                raise ValueError(f"Gemini API Hatası: {error.get('message')}")

            raise ValueError(f"Beklenmeyen Gemini Yanıtı: {json.dumps(provider_response)}")

        try:
            candidate = provider_response['candidates'][0]
            # 2. İçerik (content) kontrolü
            if 'content' in candidate and 'parts' in candidate['content']:
                content = candidate['content']['parts'][0]['text']
            else:
                finish_reason = candidate.get('finishReason', 'UNKNOWN')
                content = f"⚠️ İçerik boş döndü. (Bitiş Sebebi: {finish_reason})"

            return ChatResponse(
                model_used=model_name,
                content=content,
                finish_reason=candidate.get('finishReason')
            )
        except (KeyError, IndexError) as e:
            # Yeni hata mesajımız bu olmalı:
            raise ValueError(f"Yanıt Ayrıştırma Hatası: {str(e)}")

    async def stream_chat(self, request: ChatRequest) -> AsyncGenerator[str, None]:
        url = self.get_url(request)
        headers = self.get_headers()
        payload = self.convert_request(request)

        async with self.client.stream("POST", url, json=payload, headers=headers) as response:
            response.raise_for_status()
            async for line in response.aiter_lines():
                if line.startswith("data:"):
                    json_str = line[5:].strip()
                    if not json_str: continue
                    try:
                        chunk = json.loads(json_str)
                        if 'candidates' in chunk and chunk['candidates']:
                            content = chunk['candidates'][0].get('content')
                            if content and 'parts' in content:
                                text = content['parts'][0].get('text')
                                if text: yield text
                    except: continue

In [ ]:
# @title Hücre 9: OpenAI Sağlayıcısı
# Standart Chat Completions API yapısı.
%%writefile app/providers/openai.py
from app.providers.base import BaseProvider
from app.models.chat import ChatRequest, ChatResponse
from app.core.config import settings
from typing import Dict, Any, AsyncGenerator
import json

class OpenAIProvider(BaseProvider):
    def get_headers(self) -> Dict[str, str]:
        if not settings.OPENAI_API_KEY:
            raise ValueError("Yapılandırma Hatası: .env dosyasında OPENAI_API_KEY bulunamadı.")
        return {
            "Authorization": f"Bearer {settings.OPENAI_API_KEY}",
            "Content-Type": "application/json"
        }

    def get_url(self, request: ChatRequest) -> str:
        return "https://api.openai.com/v1/chat/completions"

    def convert_request(self, request: ChatRequest) -> Dict[str, Any]:
        # Bizim Message modelimizi dict'e çevir
        messages_dict = [msg.model_dump() for msg in (request.messages or [])]

        payload = {
            "model": request.model,
            "messages": messages_dict,
            "stream": request.stream,
            "temperature": request.temperature,
        }

        if request.max_tokens:
            payload["max_tokens"] = request.max_tokens

        if request.json_mode:
            payload["response_format"] = {"type": "json_object"}

        return payload

    def convert_response(self, provider_response: Dict[str, Any], model_name: str) -> ChatResponse:
        try:
            content = provider_response['choices'][0]['message']['content']
            finish_reason = provider_response['choices'][0]['finish_reason']
            return ChatResponse(
                model_used=model_name,
                content=content,
                finish_reason=finish_reason
            )
        except KeyError:
             raise ValueError(f"OpenAI Yanıt Hatası: {provider_response}")

    async def stream_chat(self, request: ChatRequest) -> AsyncGenerator[str, None]:
        url = self.get_url(request)
        headers = self.get_headers()
        payload = self.convert_request(request)

        async with self.client.stream("POST", url, json=payload, headers=headers) as response:
            response.raise_for_status()
            async for line in response.aiter_lines():
                if line.startswith("data: "):
                    data_str = line[6:].strip()
                    if data_str == "[DONE]":
                        break
                    try:
                        chunk_data = json.loads(data_str)
                        delta = chunk_data['choices'][0]['delta'].get('content')
                        if delta:
                            yield delta
                    except json.JSONDecodeError:
                        continue

In [ ]:
# @title Hücre 10: Sağlayıcı Kayıt Defteri (Registry)
# Hangi modelin hangi sağlayıcıyı kullanacağını belirler.
%%writefile app/providers/__init__.py
from .gemini import GeminiProvider
from .openai import OpenAIProvider
import httpx

# Performans için tek bir HTTP istemcisi (Connection Pooling)
client = httpx.AsyncClient(timeout=60.0)

# Model öneklerine (prefix) göre sağlayıcı eşleştirmesi
PROVIDER_MAP = {
    "gemini-": GeminiProvider(client),
    "gpt-": OpenAIProvider(client),
    # "claude-": AnthropicProvider(client), # Gelecekte
}

def get_provider(model_name: str):
    """Model adına bakarak uygun Provider sınıfını döndürür."""
    for prefix, provider in PROVIDER_MAP.items():
        if model_name.startswith(prefix):
            return provider
    return None

In [ ]:
# @title Hücre 11: Akıllı Yönlendirme Servisi (Logic)
# Default değerleri atar, 'oto' modunu yönetir ve format dönüşümlerini yapar.
%%writefile app/services/router_service.py
from app.models.chat import ChatRequest, ChatResponse, Message
from app.providers import get_provider
from fastapi import HTTPException
import httpx
import json
import time
from typing import AsyncGenerator

# Varsayılan "oto" modeli
DEFAULT_MODEL = "gemini-2.5-flash"

def normalize_request(request: ChatRequest) -> ChatRequest:
    """
    Kullanıcıdan gelen eksik veya 'kolay mod' isteklerini
    standart, tam bir isteğe dönüştürür.
    """
    # 1. Model 'oto' veya boş ise varsayılanı seç
    if not request.model or request.model == "oto":
        request.model = DEFAULT_MODEL

    # 2. 'messages' boş ama 'prompt' doluysa (Kolay Mod), mesaja çevir
    if not request.messages and request.prompt:
        request.messages = [
            Message(role="user", content=request.prompt)
        ]

    # 3. İkisi de yoksa hata ver
    if not request.messages:
        raise HTTPException(
            status_code=400,
            detail="Eksik Parametre: 'messages' listesi veya 'prompt' metni girilmelidir."
        )

    return request

async def handle_chat_request(request: ChatRequest) -> ChatResponse:
    """Tekil yanıt (Non-stream) işlemi."""
    request = normalize_request(request)
    provider = get_provider(request.model)

    if not provider:
        raise HTTPException(status_code=400, detail=f"Desteklenmeyen Model: '{request.model}' için bir sağlayıcı bulunamadı.")

    try:
        # Sağlayıcıya özel formata çevir ve isteği at
        provider_url = provider.get_url(request)
        provider_headers = provider.get_headers()
        provider_payload = provider.convert_request(request)

        response = await provider.client.post(provider_url, json=provider_payload, headers=provider_headers)
        response.raise_for_status()

        # Cevabı standart formata çevir
        return provider.convert_response(response.json(), request.model)

    except httpx.HTTPStatusError as e:
        # Dış API hatalarını temiz bir şekilde göster
        error_detail = e.response.text
        raise HTTPException(status_code=e.response.status_code, detail=f"Sağlayıcı Hatası ({request.model}): {error_detail}")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Sunucu İçi Hata: {str(e)}")

async def handle_chat_stream(request: ChatRequest) -> AsyncGenerator[str, None]:
    """Akışlı yanıt (Stream) işlemi - OpenAI uyumlu format."""
    request = normalize_request(request)
    provider = get_provider(request.model)

    stream_id = f"chatcmpl-{int(time.time())}"

    if not provider:
        yield f"data: {json.dumps({'error': f'Model {request.model} bulunamadı'})}\n\n"
        return

    try:
        async for chunk_text in provider.stream_chat(request):
            # OpenAI standart stream formatı
            response_chunk = {
                "id": stream_id,
                "object": "chat.completion.chunk",
                "created": int(time.time()),
                "model": request.model,
                "choices": [{
                    "index": 0,
                    "delta": {"content": chunk_text},
                    "finish_reason": None
                }]
            }
            yield f"data: {json.dumps(response_chunk)}\n\n"

    except Exception as e:
        error_json = {"error": f"Akış Hatası: {str(e)}"}
        yield f"data: {json.dumps(error_json)}\n\n"

    yield "data: [DONE]\n\n"

In [ ]:
# @title Hücre 12: Sohbet Endpoint (Router)
# Dış dünyaya açılan kapı. Hem JSON hem Stream dönebilir.
%%writefile app/routers/chat.py
from fastapi import APIRouter, Depends
from fastapi.responses import StreamingResponse, JSONResponse
from app.models.chat import ChatRequest, ChatResponse
from app.core.security import get_api_key
from app.services import router_service

router = APIRouter()

@router.post(
    "/v1/chat/completions",
    dependencies=[Depends(get_api_key)],
    summary="Metin Üretimi (Chat)",
    description="Metin veya sohbet geçmişi göndererek AI yanıtı alın. 'stream=True' ile anlık akış destekler.",
    response_model=None # Union dönüş tipi hatasını önlemek için
)
async def generate_chat(request: ChatRequest):
    # İsteği servise ilet
    if request.stream:
        return StreamingResponse(
            router_service.handle_chat_stream(request),
            media_type="text/event-stream"
        )
    else:
        response = await router_service.handle_chat_request(request)
        return JSONResponse(content=response.model_dump())

# --- YER TUTUCULAR (Gelecek Özellikler) ---
# @router.post("/v1/audio/speech") -> TTS eklenecek
# @router.post("/v1/images/generations") -> Resim üretimi eklenecek
# @router.post("/v1/audio/transcriptions") -> Ses-Metin (Whisper) eklenecek

In [ ]:
# @title Hücre 13: Ana Uygulama (Main)
# FastAPI uygulamasını başlatır ve router'ları bağlar.
%%writefile app/main.py
from fastapi import FastAPI
from app.routers import chat

app = FastAPI(
    title="Gelişmiş AI Gateway API",
    description="Gemini ve OpenAI modellerini tek çatı altında toplayan, optimize edilmiş API ağ geçidi.",
    version="1.0.0"
)

# Router'ları ekle
app.include_router(chat.router, prefix="/api", tags=["Text Generation"])

@app.get("/", tags=["Health"])
def health_check():
    return {"status": "active", "message": "AI Gateway çalışıyor. /docs adresinden dökümantasyona bakabilirsiniz."}

In [ ]:
# @title Hücre 14: Sunucuyu Başlat (Runner)
# Ngrok tünelini açar ve Uvicorn sunucusunu başlatır.
import sys
import os
sys.path.insert(0, os.path.abspath('.'))
from pyngrok import ngrok
import uvicorn
import asyncio

# --- NGROK TOKEN ---
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")  # 🔑 Secrets bölümünden okunur
# -------------------

try:
    ngrok.kill() # Eski tünelleri temizle
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"✅ API Yayında: {public_url}")
    print(f"📄 Dökümantasyon: {public_url}/docs")
except Exception as e:
    print(f"❌ Ngrok Hatası: {e}")

if 'public_url' in globals():
    config = uvicorn.Config("app.main:app", host="127.0.0.1", port=8000, log_level="info", reload=True)
    server = uvicorn.Server(config)
    await server.serve()

In [ ]:
# @title Hücre 15: Vision Veri Modelleri
# Görüntü analizi için gerekli veri yapıları.
%%writefile app/models/vision.py
from pydantic import BaseModel, Field
from typing import List, Optional, Union

class VisionRequest(BaseModel):
    # Kolay Mod: Sadece URL ve soru
    image_url: Optional[str] = Field(None, description="Analiz edilecek resmin linki.")
    prompt: Optional[str] = Field("Bu resimde ne var?", description="Resim hakkında sorulan soru.")

    # Gelişmiş Mod: Model seçimi ve ayarlar
    model: Optional[str] = Field("oto", description="Kullanılacak Vision modeli.")
    max_tokens: Optional[int] = Field(300, description="Cevap uzunluğu.")
    temperature: Optional[float] = Field(0.5, description="Yaratıcılık seviyesi.")

class VisionResponse(BaseModel):
    model_used: str
    description: str
    usage: Optional[dict] = None

In [ ]:
# @title Hücre 16: Vision Sağlayıcı Şablonu
# Tüm Vision modellerinin uyması gereken kurallar.
%%writefile app/providers/base_vision.py
from abc import ABC, abstractmethod
from app.models.vision import VisionRequest, VisionResponse
import httpx

class BaseVisionProvider(ABC):
    def __init__(self, client: httpx.AsyncClient):
        self.client = client

    @abstractmethod
    async def analyze_image(self, request: VisionRequest) -> VisionResponse:
        """Bir resmi analiz eder ve metin açıklaması döner."""
        pass

In [ ]:
# @title Hücre 17: OpenAI Vision Sağlayıcısı (GPT-4o)
# OpenAI, resim URL'lerini doğrudan kabul ettiği için işlem basittir.
%%writefile app/providers/openai_vision.py
from app.providers.base_vision import BaseVisionProvider
from app.models.vision import VisionRequest, VisionResponse
from app.core.config import settings
import json

class OpenAIVisionProvider(BaseVisionProvider):
    def get_headers(self):
        return {
            "Authorization": f"Bearer {settings.OPENAI_API_KEY}",
            "Content-Type": "application/json"
        }

    async def analyze_image(self, request: VisionRequest) -> VisionResponse:
        url = "https://api.openai.com/v1/chat/completions"

        # OpenAI Vision Formatı
        payload = {
            "model": request.model,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": request.prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": request.image_url}
                        }
                    ]
                }
            ],
            "max_tokens": request.max_tokens
        }

        response = await self.client.post(url, json=payload, headers=self.get_headers())
        response.raise_for_status()
        data = response.json()

        return VisionResponse(
            model_used=request.model,
            description=data['choices'][0]['message']['content']
        )

In [ ]:
# @title Hücre 18: Gemini Vision Sağlayıcısı (Base64 Dönüştürücülü)
# Gemini URL yerine Base64 istediği için, URL'yi indirip dönüştüren bir ara katman ekliyoruz.
%%writefile app/providers/gemini_vision.py
from app.providers.base_vision import BaseVisionProvider
from app.models.vision import VisionRequest, VisionResponse
from app.core.config import settings
import base64
import httpx

class GeminiVisionProvider(BaseVisionProvider):
    def get_url(self, model_name):
        return f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={settings.GEMINI_API_KEY}"

    async def _url_to_base64(self, url: str) -> tuple[str, str]:
        """Resim URL'sini indirir ve Base64 formatına çevirir."""
        # Resmi indir
        response = await self.client.get(url)
        response.raise_for_status()

        # Mime tipini al (örn: image/jpeg)
        mime_type = response.headers.get("content-type", "image/jpeg")

        # Byte verisini Base64 string'e çevir
        b64_data = base64.b64encode(response.content).decode("utf-8")
        return mime_type, b64_data

    async def analyze_image(self, request: VisionRequest) -> VisionResponse:
        if not settings.GEMINI_API_KEY:
            raise ValueError("GEMINI_API_KEY eksik.")

        # 1. URL'yi Base64'e çevir (Gemini için gerekli)
        mime_type, b64_data = await self._url_to_base64(request.image_url)

        # 2. Gemini Payload Hazırla
        payload = {
            "contents": [{
                "parts": [
                    {"text": request.prompt},
                    {
                        "inline_data": {
                            "mime_type": mime_type,
                            "data": b64_data
                        }
                    }
                ]
            }],
            "generationConfig": {
                "maxOutputTokens": request.max_tokens,
                "temperature": request.temperature
            }
        }

        # 3. İsteği Gönder
        api_url = self.get_url(request.model)
        response = await self.client.post(api_url, json=payload, headers={"Content-Type": "application/json"})
        response.raise_for_status()
        data = response.json()

        # 4. Yanıtı İşle
        try:
            description = data['candidates'][0]['content']['parts'][0]['text']
            return VisionResponse(
                model_used=request.model,
                description=description
            )
        except (KeyError, IndexError):
            raise ValueError(f"Gemini Vision Hatası: {data}")

In [ ]:
# @title Hücre 19: Vision Kayıt Defteri
# Hangi modelin hangi vision sağlayıcıyı kullanacağını seçer.
%%writefile app/providers/vision_registry.py
from .gemini_vision import GeminiVisionProvider
from .openai_vision import OpenAIVisionProvider
import httpx

client = httpx.AsyncClient(timeout=60.0)

VISION_PROVIDER_MAP = {
    "gemini-": GeminiVisionProvider(client),
    "gpt-": OpenAIVisionProvider(client),
}

def get_vision_provider(model_name: str):
    for prefix, provider in VISION_PROVIDER_MAP.items():
        if model_name.startswith(prefix):
            return provider
    return None

In [ ]:
# @title Hücre 20: Vision Servisi (Yönlendirici)
# Default değerleri atar ve doğru sağlayıcıya yönlendirir.
%%writefile app/services/vision_service.py
from app.models.vision import VisionRequest, VisionResponse
from app.providers.vision_registry import get_vision_provider
from fastapi import HTTPException

# Vision için varsayılan model (Gemini 2.5 Flash çok hızlı ve ücretsiz)
DEFAULT_VISION_MODEL = "gemini-2.5-flash"

async def process_vision_request(request: VisionRequest) -> VisionResponse:
    # 1. Oto model seçimi
    if request.model == "oto" or not request.model:
        request.model = DEFAULT_VISION_MODEL

    # 2. Resim URL kontrolü
    if not request.image_url:
        raise HTTPException(status_code=400, detail="Lütfen bir 'image_url' sağlayın.")

    # 3. Sağlayıcıyı bul
    provider = get_vision_provider(request.model)
    if not provider:
        raise HTTPException(status_code=400, detail=f"Desteklenmeyen Vision Modeli: {request.model}")

    # 4. İşlemi yap
    try:
        return await provider.analyze_image(request)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Vision İşleme Hatası: {str(e)}")

In [ ]:
# @title Hücre 21: Vision Endpoint (Router)
# /api/v1/vision/analyze adresini açar.
%%writefile app/routers/vision.py
from fastapi import APIRouter, Depends
from app.models.vision import VisionRequest, VisionResponse
from app.core.security import get_api_key
from app.services import vision_service

router = APIRouter()

@router.post(
    "/v1/vision/analyze",
    dependencies=[Depends(get_api_key)],
    response_model=VisionResponse,
    summary="Görsel Analizi (Vision)",
    description="Bir resim URL'si gönderin, AI size resimde ne olduğunu anlatsın."
)
async def analyze_image(request: VisionRequest):
    return await vision_service.process_vision_request(request)

### Blok 8: Sistemi Güncelleme (Main & Runner)

Son olarak, yeni eklediğimiz "Vision" özelliğini ana uygulamaya (`app/main.py`) tanıtmamız ve sunucuyu yeniden başlatmamız gerekiyor.

In [ ]:
# @title Hücre 22: Ana Uygulamayı Güncelleme (Vision Eklendi)
%%writefile app/main.py
from fastapi import FastAPI
from app.routers import chat
from app.routers import vision # Yeni router

app = FastAPI(
    title="AI Gateway API (Text + Vision)",
    description="Metin ve Görsel işleme yeteneklerine sahip bütünleşik AI API.",
    version="1.1.0"
)

# Router'ları bağla
app.include_router(chat.router, prefix="/api", tags=["Text Generation"])
app.include_router(vision.router, prefix="/api", tags=["Vision Analysis"])

@app.get("/", tags=["Health"])
def health_check():
    return {"status": "active", "capabilities": ["text", "vision"]}

In [ ]:
# @title Hücre 23: Sunucuyu (Yeniden) Başlat
# Ngrok token'ınız Hücre 14'te zaten tanımlıydı, burada tekrar yazmaya gerek yok.
import sys
import os
sys.path.insert(0, os.path.abspath('.'))
from pyngrok import ngrok
import uvicorn

# Eğer token tanımlı değilse uyarı ver (Hücre 14 çalıştırılmış olmalı)
try:
    if 'NGROK_AUTH_TOKEN' not in globals():
        print("UYARI: NGROK_AUTH_TOKEN bulunamadı. Lütfen Hücre 14'teki token'ı buraya kopyalayın veya Hücre 14'ü tekrar çalıştırın.")
    else:
        ngrok.kill()
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        public_url = ngrok.connect(8000)
        print(f"✅ API Güncellendi ve Yayında: {public_url}")
        print(f"📄 Yeni Dökümantasyon: {public_url}/docs")

        config = uvicorn.Config("app.main:app", host="127.0.0.1", port=8000, log_level="info", reload=True)
        server = uvicorn.Server(config)
        await server.serve()
except Exception as e:
    print(f"Başlatma Hatası: {e}")

### Nasıl Test Edeceğiz?

Sunucu yeniden başladıktan sonra (Hücre 23), aşağıdaki basit Python kodunu Visual Studio Code'da çalıştırarak Vision özelliğini test edebilirsiniz.

```python
# test_vision.py
import httpx

API_URL = "https://....ngrok-free.dev/api/v1/vision/analyze" # Sizin URL
API_KEY = "<INTERNAL_API_KEY>"

headers = {"Authorization": f"Bearer {API_KEY}"}

# Test Resmi: Bir kedi resmi
payload = {
    "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg",
    "prompt": "Bu resimde ne var? Detaylı anlat.",
    "model": "oto" # Otomatik olarak Gemini seçilecek
}

print("Resim analiz ediliyor...")
response = httpx.post(API_URL, json=payload, headers=headers, timeout=60)

if response.status_code == 200:
    print("✅ BAŞARILI!")
    print(response.json()['description'])
else:
    print(f"❌ HATA: {response.text}")
```